# Cosmological zoom-in the HAST pipeline — Part 1: parent box and halo selection

This series takes you through the full pipeline to run a cosmological zoom-in simulation of a Milky-Way-mass halo with RAMSES using the python tool hast to select and decontaminate halos:

| Step | Code | What it does | Part |
|---|---|---|---|
| 1 | MUSIC2 | Uniform-resolution ICs of the parent box | **1** |
| 2 | RAMSES | Dark-matter-only parent run with the clump finder | **1** |
| 3 | hast | Select an isolated halo and trace its Lagrangian region back to $z_\mathrm{init}$ | **1** |
| 4 | MUSIC2 | Dark-matter-only zoom ICs around that region | 2 |
| 5 | RAMSES | Dark-matter-only zoom run | 2 |
| 6 | hast | Decontaminate: recompute the Lagrangian region from the zoom run | 2 |
| 7 | MUSIC2 | Zoom ICs with baryons around the clean region | 3 |
| 8 | RAMSES | Hydrodynamical zoom run | 3 |

Steps 1–2 follow the [Cosmological-Volume](https://ramses-tutorials.readthedocs.io/en/latest/) tutorial; please go through it first. Here we only explain what changes, and why, when the box is meant to host a zoom.

**Laptop vs production.** Higher resolution simulations are not meant to be run in a laptop but most of the simulations here are. The zoom parts (2 and 3) use `levelmax = 9` (effective $512^3$) so that everything runs on a laptop. For completeness, each part ends with a *Production settings* box.

**self-contained** all config files are in the folder MyFirstZoom.

**MW-references** originaly this pipeline has been designed to build MW-like halos but there is no reason why it could be used for any halo mass by switching to the targe mass in any step were the MW is used to justify a particular choice.

## 0. Before we start

Install RAMSES as in section 0 of the Cosmological-Volume tutorial, and export `RAMSESPATH`.

For MUSIC2, the 4th step might require to clone the [luturonunca/MUSIC2](https://github.com/luturonunca/MUSIC2) fork instead of the upstream repository (This should be solved). It fixes a problem while reading the convex-hull refinement region used for zooms in Part 2: with upstream MUSIC2 the refinement mask can come out empty, and RAMSES then runs a uniform box instead of a zoom. Build it as in Cosmological-Volume and export `MUSICPATH`:

```bash
git clone https://github.com/luturonunca/MUSIC2.git
mkdir MUSIC2/build && cd MUSIC2/build
cmake .. && make
export MUSICPATH=$(pwd)/
```

### 0.1. hast

hast is a python tool written by Valentin Perret, it is meant for Halo selection and decontamination. A newer version relying on yt is available at [hast](https://github.com/luturonunca/hast3).

```bash
git clone https://github.com/luturonunca/hast3.git
export HASTPATH=$(pwd)/hast3/
export PYTHONPATH=$HASTPATH:$PYTHONPATH   # the example scripts do `import hast`
```

**Requirements.** Everything in the ramses-tutorials [list of requirements](https://ramses-tutorials.readthedocs.io/en/latest/Setup/general_requirements.html) (which includes `yt`), plus two packages used by hast:

```
- seaborn
- scikit-learn
```

## 1. Parent box initial conditions

### 1.1. MUSIC parameter file

The file is `MyFirstZoom/ParentBox/ICs/music.conf`. It is similar to  `MyFirstCosmoSim/ICs/music.conf` from the Cosmological-Volume tutorial with some small differences:

| Parameter | Cosmological-Volume | Here | Why |
|---|---|---|---|
| `boxlength` | 400 | 25 | The particle mass is $m_p \propto L^3$. At $128^3$, a 400 Mpc/h box gives $m_p \approx 4\times10^{12}\,M_\odot$, so Milky-Way-mass halos are not even one particle. In 25 Mpc/h, $m_p \approx 9.6\times10^{8}\,M_\odot$ and a $10^{12}\,M_\odot$ halo has $\sim 10^3$ particles: enough for the clump finder to find it. |
| `zstart` | 50 | 100 | The zoom ICs (Part 2) have much smaller cells, so they must start earlier for the initial displacements to stay small compared with the particle spacing. The parent box uses the same `zstart` so that $z_\mathrm{init}$ of the parent and the zoom are the same. |
| `seed[7]` | 12345 | 123456 | Any value works, but **it must be copied unchanged to every zoom of this box**. The zoom reproduces the parent's large-scale modes only if the level-7 seed is the same. |
| `padding`, `overlap`, `ref_center`, `ref_extent` | set | `0` / absent | Unigrid ICs (`levelmin = levelmax`): there is no refined region, so these have no effect. |
| Cosmology | Planck 2018 + BAO | Planck 2018 | Same as our production runs. |

Everything else (`grafic2` format, `ramses_nml`, `transfer = eisenstein`, `use_2LPT = yes`) is as in Cosmological-Volume.

### 1.2. Running MUSIC

```bash
cd MyFirstZoom/ParentBox/ICs
$MUSICPATH/MUSIC music.conf
```

If all goes well this writes `ic_box_dm/level_007/` and the log `music.conf_log.txt`.

## 2. Parent dark-matter-only run

### 2.1. RAMSES namelist
If in the music.conf file `ramses_nml    = yes` then among the files generated by MUSIC there will be a file called ramses.nml that have useful parameter related to the chosen resolution.
For the Dark Matter Only (DMO) run of the parent box the namelist file is `MyFirstZoom/ParentBox/DarkMatterRun/ramses.nml`. Also based on the `MyFirstCosmoSim/DarkMatterRun/ramses.nml` but with the following differences:

| Parameter | Cosmological-Volume | Here | Why |
|---|---|---|---|
| `clumpfind` + `&CLUMPFIND_PARAMS` | absent (halos found with yt afterwards) | `.true.` | hast reads the RAMSES clump finder files `clump_*.txt*` written with each output. |
| `levelmax` | 15 | 7 | Unigrid run: the parent is only used to find halos and locate them at $z_\mathrm{init}$, not to resolve them. Masses measured here are approximate due to the low resolution. |
| Outputs | `aout = 0.5, 1.0` | `delta_aout = 0.025`, `aend = 1.0` | Gives `output_00001` at $z_\mathrm{init}=100$ and `output_00041` at $z=0$ instead of only 2 ouputs, requires more disc space. hast needs both: the halo at $z=0$ and its particles at $z_\mathrm{init}$. |
| `nremap` | 1 | 20 | Load balancing less often. Performance only. |
| `ngridtot`, `nparttot` | 3e6, 3e6 | 1e6, 1e7 | Total memory, split evenly over the MPI tasks. There are only $128^3 \approx 2.1\times10^6$ particles, but as halos form one task can hold several times its share, so `nparttot` has a large margin. |

`&REFINE_PARAMS` is kept from production but has no effect with `levelmax = levelmin`.

**Clump finder thresholds.** `density_threshold = 8` (in units of the mean density) defines where clumps can exist. `relevance_threshold = 3` and `saddle_threshold = 20` decide when two peaks are kept apart or merged. `mass_threshold = 10` (in particles) drops the smallest clumps.

### 2.2. Running RAMSES

```bash
cd MyFirstZoom/ParentBox/DarkMatterRun
mpirun -np 4 $RAMSESPATH/ramses3d ramses.nml > run.log
```

Check that `output_00001` and `output_00041` exist and contain `clump_*.txt*` files:

In [ ]:
!ls MyFirstZoom/ParentBox/DarkMatterRun/output_00041 | head

## 3. Halo selection with hast

### 3.1. Selection parameters

An example configuration file for MW mass halos can be found in `MyFirstZoom/Selection/selection.conf`. The values defined here allow hast to:

1. read the clumps at `output_zlast` and keeps those with `min_mass` < $M$ < `max_mass`;
2. drop candidates with a neighbour heavier than `max_mass_neighb` $\times M$ within `rbuffer` (physical Mpc), so the zoom target is isolated;
3. drop candidates outside [`boundary_min`, `boundary_max`] of the box where the box begingin is 0 and the end is 1;
4. for each surviving candidate, take the particles within `rtb` $\times R_{200}$ at $z=0$ and find them in `output_zinit`. The volume sorrounding their positions at $z_\mathrm{init}$ is the **Lagrangian region** of the halo.

`rtb = 3` traces back more than the halo itself, which keeps low-resolution particles out of the halo in the zoom.

### 3.2. Running hast

```bash
cd MyFirstZoom/Selection
mkdir -p results
python $HASTPATH/examples/selection.py selection.conf | tee hastlog.log
```

### 3.3. Output: the particle lists

For each selected candidate N, hast writes `results/music_zoom_N`: a text file with three columns $x, y, z$ (box units, at $z_\mathrm{init}$). These are the vertices of the convex hull around the traced-back particles two of its projections are shown in the following figure. MUSIC reads this file in Part 2 (`region_point_file`) to decide where to refine.

`results/music_zoom.pdf` shows in two projections, the candidates and their Lagrangian regions. This plots can help pick one halo for Part 2. in the case of the MW target halo one would look an isolated halo with a compact lagrangian region. In this case halo 2 is better suited.

For the box of this tutorial, 2 of the 104 candidates in the mass window survive (see `hastlog.log` for each cut):

![Selected halos](figures/selection_map.png)

*`music_zoom.pdf`.* Projected dark-matter density of the parent box at $z=0$ (axes in kpc). The dots are the selected halos at $z=0$. The contours are their Lagrangian regions, the convex hulls of the traced-back particles, at $z_\mathrm{init}$. The Lagrangian region does not sit on top of the halo: the particles have moved since $z=100$.

A compact Lagrangian region is cheaper to zoom on and less likely to be contaminated by low-resolution particles. Halo 1's region is elongated (it spans 0.26 of the box in $y$); halo 2's spans at most 0.16. We use halo 2 in Part 2.

In [ ]:
import numpy as np

N = 2  # candidate to inspect
pts = np.loadtxt(f"MyFirstZoom/Selection/results/music_zoom_{N}")
print("hull vertices :", len(pts))
print("centroid      :", pts.mean(axis=0))
print("extent (box)  :", pts.max(axis=0) - pts.min(axis=0))


A compact Lagrangian region (small extent) gives a cheaper zoom. The extent sets how many high-resolution particles the zoom will need.
Spikes in the lagrangian region usually means that a group of particles that end in the refined region are intiially located far away but there is no guarantee that all particles close to the spike are also going to finish around the halo. This type of Lagrangian region might require several decontamination steps.

---
**Production settings.** Part 1 is identical in production. The production parent run used 128 MPI tasks. Candidates were selected with the same `selection.conf`.